This quick test actually comes from this StackOverflow comment: https://stackoverflow.com/questions/76127508/iminuit-high-dimension-multivariate-fit-problem. The [GitHub issue](https://github.com/scikit-hep/iminuit/issues/974) was closed with a resolution; you can now do it with models of $n$ independent variables.

Import libraries.

In [2]:
from iminuit import Minuit
from iminuit.cost import LeastSquares
import numpy as np

Define a function $f(x, y) = a + x^{b} + y^{c}$.

In [27]:
"""
[NOTE]: tried to use x and y as separate independent variables, but imnuit will NOT
let you do that. For example, model(x, y, a, b, c) is WRONG in this case; it interprets only 
the FIRST ARGUMENT as the INDEPENDENT VARIABLE(S) and everything following that to be 
MODEL PARAMETERS
"""

def model1(xy, a, b, c):
    x, y = xy 
    return a + (x ** b) + (y ** c)

Generate random data.

In [28]:
np.random.seed(1)
data_x = np.linspace(0, 1, 11)
data_y = np.linspace(0, 1, 11)

Check out the arrays.

In [17]:
data_x

array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ])

See if we get a single value out of the model.

In [31]:
model1([data_x, data_y], 2, 3, 10)

array([2.        , 2.001     , 2.0080001 , 2.0270059 , 2.06410486,
       2.12597656, 2.22204662, 2.37124752, 2.61937418, 3.07767844,
       4.        ])

Is this right?

In [32]:
2. + (0.4**3) + (0.4**10)

2.0641048576

One more check:

In [33]:
(2. + (0.4**3) + (0.4**10)) == model1([data_x, data_y], 2, 3, 10)[4]

True

Now make some data.

In [34]:
z_error = 0.1
data_z = model1([data_x, data_y], 2, 3, 10) + z_error * np.random.randn(len(data_x))

In [36]:
least_squares = LeastSquares([data_x, data_y], data_z, z_error, model1)

m1 = Minuit(least_squares, a = 1.0, b = 1.0, c = 1.0)
m1.migrad()

/tmp/ipykernel_3161081/526479748.py:10: RuntimeWarning: divide by zero encountered in power
  return a + (x ** b) + (y ** c)


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 16.02 (χ²/ndof = 2.0)      │              Nfcn = 144              │
│ EDM = 1.76e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   2.02    │   0.04    │            │            │         │         │       │
│ 1 │ b    │    3.4    │    0.9    │            │            │         │         │       │
│ 2 │ c    │    9.7    │    3.2    │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───┬─────────────────────────┐
│   │       a       b       c │
├───┼─────────────────────────┤
│ a │ 0.00188  0.0214 -0.0003 │
│ b │  0.0214   0.737    -1.7 │
│ c │ -0.0003    -1.7    10.2 │
└───┴─────────────────────────┘

In [37]:
least_squares_2 = LeastSquares((data_x, data_y), data_z, z_error, model1)

m2 = Minuit(least_squares_2, a = 1.0, b = 1.0, c = 1.0)
m2.migrad()

/tmp/ipykernel_3161081/526479748.py:10: RuntimeWarning: divide by zero encountered in power
  return a + (x ** b) + (y ** c)


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 16.02 (χ²/ndof = 2.0)      │              Nfcn = 144              │
│ EDM = 1.76e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   2.02    │   0.04    │            │            │         │         │       │
│ 1 │ b    │    3.4    │    0.9    │            │            │         │         │       │
│ 2 │ c    │    9.7    │    3.2    │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───┬─────────────────────────┐
│   │       a       b       c │
├───┼─────────────────────────┤
│ a │ 0.00188  0.0214 -0.0003 │
│ b │  0.0214   0.737    -1.7 │
│ c │ -0.0003    -1.7    10.2 │
└───┴─────────────────────────┘

There appears to be no difference between passing in `[x_data, y_data]` or `(x_data, y_data)` into the `LeastSquares` cost function... Alright.

In [ ]:
def model2(xyz, a, b, c, d):
    x, y, z = xyz
    return a + (x ** b) + (y ** c) + (z ** d)

np.random.seed(1)
data_x = np.linspace(0, 1, 11)
data_y = np.linspace(0, 1, 11)
data_z = np.linspace(0, 1, 11)

data_w_err = 0.1
data_w = model2([data_x,data_y, data_z], 2, 3, 10, 20) + data_w_err * np.random.randn(len(data_x))

least_squares = LeastSquares([data_x, data_y, data_z], data_w, data_w_err, model2)

m3 = Minuit(least_squares, a = 1.0, b = 1.0, c = 1.0, d = 1.0)
m3.migrad() 

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 19.43 (χ²/ndof = 3.2)      │              Nfcn = 163              │
│ EDM = 3.11e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │     Covariance FORCED pos. def.      │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   2.037   │   0.028   │            │            │         │         │       │
│ 1 │ b    │     8     │    24     │            │            │         │         │       │
│ 2 │ c    │     8     │    24     │            │            │         │         │       │
│ 3 │ d    │     8     │    24     │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───┬─────────────────────────────────────┐
│   │        a        b        c        d │
├───┼─────────────────────────────────────┤
│ a │ 0.000793   7.3e-3   7.3e-3   7.3e-3 │
│ b │   7.3e-3      579   -0.3e3   -0.3e3 │
│ c │   7.3e-3   -0.3e3      579   -0.3e3 │
│ d │   7.3e-3   -0.3e3   -0.3e3      579 │
└───┴─────────────────────────────────────┘